## Verified local full run

A complete local CPU run was made on 2026-09-25 with 100,000 points, the six-scale ladder shown below, `alpha = 1..10`, and seed `20260926`. Immutable output: `results/20260925T233947Z-7c7f769921/schweinhart_controls.json`. All 95% dimension intervals had relative width below 10%.

| Shape | Known dimension | Estimate range, alpha 1 to 10 | Best APE | Mean APE, ours | Mean APE, paper Table A1 |
|---|---:|---:|---:|---:|---:|
| Sphere | 2.000 | 2.0004 to 2.0071 | 0.000% | 0.140% | 0.144% |
| Swiss roll | 2.000 | 1.9710 to 1.9032 | 1.450% | 3.078% | 0.958% |
| Sierpinski carpet | 1.8928 | 1.8891 to 1.8775 | 0.196% | 0.742% | 0.230% |
| Menger sponge | 2.7268 | 2.6956 to 2.7212 | 0.207% | 0.961% | 1.836% |

Interpretation: the implementation recovers each known dimension well enough for a numerical control. This is **not** an exact numerical replication of their Table A1: the Swiss-roll APE is materially higher, and their unpublished ladder, point samples, and embedding construction prevent matching their raw numbers exactly.

# Schweinhart Synthetic Controls

This notebook validates the MST-growth implementation on four point clouds with known intrinsic dimensions. It does **not** analyse language and does not change the preregistered dialogue-PHD experiment.

The controls follow the set used in Appendix A of Gromov, Borodin, and Yerbolova (2024): a sphere, Swiss roll, Sierpinski carpet, and Menger sponge. The paper's exact synthetic sample-size ladder was not published, so the ladder below is an explicit adaptation rather than an exact numerical replication.

## What should happen

At sufficiently large samples, the estimate at suitable `alpha` values should be near:

| Shape | Known Hausdorff dimension |
|---|---:|
| Sphere surface | 2 |
| Swiss roll | 2 |
| Sierpinski carpet | `log(8) / log(3) ~= 1.893` |
| Menger sponge | `log(20) / log(3) ~= 2.727` |

A smoke run is useful for checking installation and reproducibility. The full run uses 100,000 points, as stated for their synthetic controls, and can take substantially longer.

The paper does not publish the raw estimated dimensions for every run. It publishes true dimensions and Appendix A Table A1 absolute percentage errors (APE). Therefore, the strongest check available from the paper is: recover the known dimension, then compare our APE curve with the published APE curve. Exact equality is not expected because the exact sample-size ladder, random samples, and high-dimensional embeddings are not disclosed.

In [ ]:
# Point this at a clone of this repository.
# If the notebook was opened from the repository, use that directory instead.
from pathlib import Path

REPO = Path('/content/LANC')
REPO_URL = ''  # Example: 'https://github.com/OWNER/LANC.git'

if not REPO.exists():
    local_candidates = [
        Path.cwd(),
        Path(r'C:/Users/masog/OneDrive/Desktop/Документы/LANC'),
    ]
    REPO = next((path for path in local_candidates if (path / 'pyproject.toml').exists()), REPO)

if not REPO.exists():
    if not REPO_URL:
        raise RuntimeError('Clone the repository first, then set REPO to its Colab path.')
    !git clone $REPO_URL $REPO

%cd $REPO
print(REPO.resolve())

In [ ]:
# Colab already supplies NumPy, SciPy, scikit-learn, and PyTorch.
# Install this checkout without replacing Colab's CUDA-enabled PyTorch.
import sys
!{sys.executable} -m pip install -q -e . --no-deps

# Confirm that the numerical packages used by the controls are available.
import numpy, scipy
print('NumPy:', numpy.__version__, '| SciPy:', scipy.__version__)

## Published Appendix A reference values

Table A1 reports APE for `alpha = 1, ..., 10`. We record the four relevant columns below. Swiss roll, Sierpinski carpet, and Menger sponge are directly comparable shapes. The paper's sphere setup uses an unpublished high-dimensional embedding construction, whereas this notebook uses an ordinary 2-sphere in 3D; treat its paper APE as context, not an equality target.

In [ ]:
# Values transcribed from Gromov et al. (2024), Appendix A, Table A1.
# They are percentage errors, not the paper's raw dimension estimates.
import pandas as pd

PAPER_A1_APE_PERCENT = {
    'sphere': [0.07, 0.03, 0.03, 0.08, 0.12, 0.16, 0.19, 0.22, 0.25, 0.29],
    'swiss_roll': [0.43, 0.43, 0.47, 0.55, 0.68, 0.85, 1.08, 1.36, 1.69, 2.04],
    'sierpinski_carpet': [0.07, 0.03, 0.03, 0.18, 0.24, 0.28, 0.32, 0.35, 0.38, 0.42],
    'menger_sponge': [1.56, 1.68, 1.76, 1.82, 1.85, 1.87, 1.90, 1.92, 1.97, 2.03],
}

pd.DataFrame(PAPER_A1_APE_PERCENT, index=range(1, 11)).rename_axis('alpha')

## Full local validation

This is the complete locally verified 100,000-point run. It can take several minutes on CPU.

In [ ]:
from agent_language_geometry.schweinhart_controls import run_schweinhart_controls

FULL_POINTS = 100_000
FULL_SIZES = [10_000, 20_000, 40_000, 60_000, 80_000, 100_000]
ALPHAS = list(range(1, 11))
SEED = 20260926

full = run_schweinhart_controls(FULL_POINTS, FULL_SIZES, ALPHAS, SEED)
print('Completed deterministic full run.')

In [ ]:
# Display one row per shape and alpha.
import pandas as pd

rows = []
CONTROL_RESULT = full
for name, result in CONTROL_RESULT.items():
    for estimate in result['alpha_estimates']:
        rows.append({
            'shape': name,
            'known_dimension': result['known_dimension'],
            'alpha': estimate['alpha'],
            'estimated_dimension': estimate['dimension'],
            'r_squared': estimate['r_squared'],
            'ci_95': estimate['dimension_ci_95'],
        })

table = pd.DataFrame(rows)
table['our_ape_percent'] = (
    (table['estimated_dimension'] - table['known_dimension']).abs()
    / table['known_dimension']
    * 100
)
table

In [ ]:
# Compare our run with the paper's published APE reference.
# This is descriptive, not a replication pass/fail test: the paper omits its exact ladder and samples.
paper_ape = pd.DataFrame(PAPER_A1_APE_PERCENT, index=range(1, 11)).rename_axis('alpha')
comparison = table.copy()
comparison['paper_ape_percent'] = [
    paper_ape.loc[int(alpha), shape] for shape, alpha in zip(comparison['shape'], comparison['alpha'])
]
comparison['ape_difference_percentage_points'] = (
    comparison['our_ape_percent'] - comparison['paper_ape_percent']
)
comparison[[
    'shape', 'alpha', 'known_dimension', 'estimated_dimension',
    'our_ape_percent', 'paper_ape_percent', 'ape_difference_percentage_points', 'r_squared'
]]

In [ ]:
# A stable region should be visually near the horizontal known-dimension line.
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
for axis, (name, result) in zip(axes.flat, CONTROL_RESULT.items()):
    subset = table[table['shape'] == name]
    axis.plot(subset['alpha'], subset['estimated_dimension'], marker='o')
    axis.axhline(result['known_dimension'], color='black', linestyle='--', label='known')
    axis.set_title(name)
    axis.set_xlabel('alpha')
    axis.set_ylabel('estimated dimension')
    axis.legend()
fig.tight_layout()

## Immutable run artifact

The full numerical output used in this executed notebook is retained as an immutable ignored run directory.

The alpha sweep and the 100,000-point scale follow the paper. The ladder is our documented adaptation because the authors do not publish theirs.

In [ ]:
RUN_DIR = REPO / 'results' / '20260925T233947Z-7c7f769921'
RESULT_PATH = RUN_DIR / 'schweinhart_controls.json'
assert RESULT_PATH.exists(), f'Missing immutable result: {RESULT_PATH}'
print(RESULT_PATH)

## Important: this is not yet the paper's language experiment

One copy of *War and Peace* cannot simply be placed into the original protocol. Gromov et al. construct a `word x text` matrix across thousands of separate literary works. With one novel there is only one document column. Splitting the novel into chapters or paragraphs would create a useful adapted experiment, but it would not be a reproduction and should not be expected to yield their values near 9.

Exact reproduction of their language tables requires their corpus manifests, text editions, stop-word lists, entity/number replacement rules, lemmatiser versions, sample-size ladder, and random seeds.